# 02c - Keputusan juara & bundel export
Lanjutan `02b_robustness.ipynb`. Tiga hal dikerjakan dan didokumentasikan di sini:
1. **Keputusan resmi** juara dan challenger, termasuk satu penyimpangan dari aturan awal beserta alasannya.
2. **Bundel export** model yang portabel (tidak bergantung pada notebook ini maupun versi sklearn/Python tertentu).
3. **Uji paritas** di proses Python yang bersih, membuktikan bundel menghasilkan skor yang sama dengan model asli (termasuk saat fitur kosong), plus pengukuran latensi per request.

Data TEST tidak disentuh di notebook ini.

## 1. Keputusan juara
**Aturan yang ditetapkan sebelum melihat hasil (02b):** skor robust = 0,5 x PR-AUC bersih + 0,5 x rata-rata PR-AUC skenario rusak; bila beberapa model berada dalam 1 SE, pilih yang paling sederhana.
Aturan itu menghasilkan **HGB `mask+grup+noise`** sebagai juara dan XGB `mask` sebagai challenger (selisih skor robust 0,0016, jauh di bawah 1 SE = 0,0135, yaitu seri).

**Penyimpangan dari aturan, dan alasannya.** Aturan awal tidak memuat *stabilitas antar random seed*. Evaluasi akhir 02b (bagian 8) menunjukkan (DEV-OOT Jul-Sep, 3 seed model + augmentasi):

| Konfigurasi | PR-AUC bersih | PR-AUC saat 20% nilai hilang |
|---|---|---|
| XGB `mask` | 0,669 ± 0,015 | 0,532 ± 0,017 |
| HGB `mask+grup+noise` | 0,612 ± 0,047 | 0,391 ± **0,138** |

HGB hasil augmentasi sangat bergantung pada seed (simpangan 0,138 pada skenario yang justru ingin kita perbaiki). Itu bertentangan dengan tujuan 'model robust'.
Tambahan: PR-AUC DEV-OOT bersih XGB `mask` sedikit lebih tinggi (0,654 vs 0,622; selisih bootstrap median +0,033, CI95 -0,010..+0,068, **tidak beda nyata**), dan format XGBoost (JSON native) lebih mudah diekspor daripada HGB sklearn dengan fitur kategorikal.

**Keputusan: juara = XGB `mask`, challenger = HGB `mask+grup+noise`.** Karena ini keputusan setelah melihat hasil, ia dicatat sebagai *override* eksplisit; aturan seleksi untuk iterasi berikutnya perlu menambahkan kriteria stabilitas seed.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")      # notebooks use paths relative to the project root
print("working directory:", os.getcwd())

In [1]:
import os, sys, json, time, shutil, subprocess, warnings, joblib, hashlib
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
sys.path.insert(0, os.getcwd()); import splits as S
import xgboost as xgb

ART, BUNDLE = "artifacts", "bundle/fraud_xgb_mask_v1"
sel = json.load(open(f"{ART}/selection_robust.json"))
cv = pd.read_csv(f"{ART}/robustness_cv.csv")
fs = cv.assign(k=np.where(cv.scenario == "clean", "bersih", "rusak")).groupby(["config", "fold", "k"]).PR_AUC.mean().unstack("k")
fs["robust"] = 0.5 * fs.bersih + 0.5 * fs.rusak
tab = fs.groupby("config").agg(bersih=("bersih", "mean"), rusak=("rusak", "mean"), robust=("robust", "mean"), se=("robust", lambda s: s.std() / np.sqrt(len(s)))).round(4)
print("frozen oleh 02b:", {k: v["config"] for k, v in sel["frozen"].items()}); tab.loc[["XGB|mask", "HGB|mask+grup+noise", "XGB|clean", "HGB|clean", "LR|clean"]]

frozen oleh 02b: {'champion': 'HGB|mask+grup+noise', 'challenger': 'XGB|mask'}


,bersih,rusak,robust,se
config,,,,
XGB|mask,0.7618,0.5905,0.6761,0.0219
HGB|mask+grup+noise,0.7514,0.6040,0.6777,0.0135
XGB|clean,0.7563,0.4480,0.6022,0.0198
HGB|clean,0.7625,0.5092,0.6359,0.0184
LR|clean,0.3225,0.2187,0.2706,0.0242


In [2]:
assert sel["frozen"]["challenger"]["config"] == "XGB|mask", "model yang diekspor harus XGB|mask"
final = dict(champion=dict(config="XGB|mask", model="XGB", train_recipe="mask", source_model_file="models/baseline_XGB_mask.joblib",
                           thr_suspicious=sel["frozen"]["challenger"]["thr_suspicious"], thr_high_risk=sel["frozen"]["challenger"]["thr_high_risk"],
                           dev_oot_pr_auc_clean=sel["frozen"]["challenger"]["oot_pr_auc_clean"], dev_oot_pr_auc_miss20=sel["frozen"]["challenger"]["oot_pr_auc_miss20"]),
             challenger=dict(config="HGB|mask+grup+noise", model="HGB", source_model_file="models/baseline_HGB_mask+grup+noise.joblib"),
             decision=dict(rule_outcome="HGB|mask+grup+noise (seri dalam 1 SE, paling sederhana)", final="XGB|mask", override=True,
                           reason="HGB hasil augmentasi tidak stabil antar seed (std PR-AUC miss20 0.138 vs 0.017 untuk XGB mask); DEV-OOT bersih XGB sedikit lebih tinggi (tidak signifikan); format export XGB lebih mudah",
                           follow_up="tambahkan kriteria stabilitas seed ke aturan seleksi pada iterasi berikutnya"))
json.dump(final, open(f"{ART}/final_selection.json", "w"), indent=1); final["decision"]

{'rule_outcome': 'HGB|mask+grup+noise (seri dalam 1 SE, paling sederhana)',
 'final': 'XGB|mask',
 'override': True,
 'reason': 'HGB hasil augmentasi tidak stabil antar seed (std PR-AUC miss20 0.138 vs 0.017 untuk XGB mask); DEV-OOT bersih XGB sedikit lebih tinggi (tidak signifikan); format export XGB lebih mudah',
 'follow_up': 'tambahkan kriteria stabilitas seed ke aturan seleksi pada iterasi berikutnya'}

## 2. Rancangan bundel export
**Masalah file `.joblib`:** ia hanya objek Python yang bergantung pada versi Python/sklearn/XGBoost yang sama dan pada kelas `Model` di notebook. Tidak bisa dipakai layanan lain (Java, SAS, dsb.).

**Bundel (`bundle/fraud_xgb_mask_v1/`):**
| File | Fungsi |
|---|---|
| `model.json` | model XGBoost dalam format JSON native (dapat dimuat Python, R, Java/xgboost4j, C++), termasuk pemisahan kategori |
| `schema.json` | kontrak input: nama fitur, urutan, tipe, daftar kategori, grup fitur; koreksi prior; ambang pita |
| `score.py` | pemuat dan penskor mandiri (hanya xgboost, numpy, pandas) yang menjadi dasar API di k8s |
| `requirements.txt` | versi paket yang diuji |
| `MODEL_CARD.md` | tujuan, data, hasil, keterbatasan, dan pemantauan |
| `parity_report.json` | bukti uji paritas dan latensi |

**Aturan layanan yang tertanam di `score.py`:**
* Fitur kosong/tidak ada pada request = `null` = *tidak tersedia* (model dilatih untuk itu pada 02b). Kategori yang tidak dikenal diperlakukan sebagai tidak tersedia.
* `score` = probabilitas terkoreksi prior (karena negatif di-down-sample 3% saat training). Ambang pita berlaku pada `score`, bukan `score_raw`.
* `quality = low` bila >40% fitur kosong atau satu grup fitur penuh tidak tersedia, sehingga decision rule hilir dapat jatuh ke aturan deterministik §6.3.
* `reasons`: tiga fitur penyumbang skor terbesar (SHAP) untuk reason code (§6.1).

In [3]:
class Model:                 # kerangka kosong hanya agar pickle dari 02b dapat dibuka; yang dipakai adalah atribut .m (XGBClassifier)
    pass
orig = joblib.load(final["champion"]["source_model_file"])
clf = orig.m; booster = clf.get_booster()
assert list(booster.feature_names) == sel["features"], "urutan fitur model harus sama dengan daftar fitur beku"
print("model:", type(clf).__name__, "| pohon:", len(booster.get_dump()), "| fitur:", len(sel["features"]), "| params:", {k: clf.get_params()[k] for k in ("n_estimators", "learning_rate", "max_depth")})

dev = S.load_dev()
for c in S.CAT: dev[c] = dev[c].astype("category")
GROUPS = {   # sama dengan 02b
  "device": [c for c in sel["features"] if c.startswith(("new_device", "cred_change", "device_cnt"))],
  "geo": [c for c in sel["features"] if c.startswith("geo_mismatch")],
  "merchant": [c for c in sel["features"] if c.startswith("mix_")],
  "payee": [c for c in sel["features"] if c.startswith(("new_payee", "distinct", "payee_concentration", "top_payee", "ftp_large"))],
  "malam": [c for c in sel["features"] if c.startswith("night_share")],
  "saldo": [c for c in sel["features"] if c.startswith(("passthrough", "out_to_in", "out_to_balance", "min_balance", "balance", "days_low"))],
  "profil": [c for c in ("declared_income", "age_ord", "gender", "region", "occupation", "segment", "shared_device_flag", "tenure_months", "kyc_age_months") if c in sel["features"]]}

os.makedirs(BUNDLE, exist_ok=True)
booster.save_model(f"{BUNDLE}/model.json")
schema = dict(version="1.0", model="XGB|mask", created=time.strftime("%Y-%m-%d"), neg_keep=sel["neg_keep"], alert_rate=sel["alert_rate"],
              thresholds=dict(suspicious=final["champion"]["thr_suspicious"], high_risk=final["champion"]["thr_high_risk"],
                              note="berlaku pada 'score' (terkoreksi prior); suspicious = presisi >= 5% di DEV-OOT, high_risk = presisi >= 50%"),
              null_policy="fitur kosong/tidak ada = tidak tersedia; kategori tidak dikenal = tidak tersedia",
              features=[dict(name=f, type="categorical" if f in S.CAT else "numeric", **({"categories": [str(x) for x in dev[f].cat.categories]} if f in S.CAT else {})) for f in sel["features"]],
              groups=GROUPS)
json.dump(schema, open(f"{BUNDLE}/schema.json", "w"), indent=1, ensure_ascii=False)
shutil.copy("deploy/score.py", f"{BUNDLE}/score.py")
open(f"{BUNDLE}/requirements.txt", "w").write(f"xgboost=={xgb.__version__}\nnumpy=={np.__version__}\npandas=={pd.__version__}\n")
print(sorted(os.listdir(BUNDLE)), "| model.json", f"{os.path.getsize(f'{BUNDLE}/model.json') / 1e3:.0f} KB")

model: XGBClassifier | pohon: 300 | fitur: 68 | params: {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 5}


['model.json', 'requirements.txt', 'schema.json', 'score.py'] | model.json 1005 KB


## 3. Uji paritas di proses yang bersih
Bundel harus memberi skor yang **sama** dengan model asli. Ujinya:
* Data: 5.000 baris DEV-OOT acak, ditambah versi yang 20% selnya dikosongkan, ditambah baris dengan kategori tidak dikenal.
* Skor referensi dihitung di notebook ini dengan model asli (`joblib` + `XGBClassifier`).
* Skor bundel dihitung di **proses Python terpisah** (`python -I`, hanya memuat bundel) lalu dibandingkan. Toleransi selisih maksimum 1e-6.
* Di proses yang sama, latensi 200 request tunggal (format dict/JSON, `None` untuk fitur kosong) diukur. Target dokumen: di bawah 150 ms untuk keseluruhan gate; tahap model idealnya jauh di bawah itu.

In [4]:
oot = S.dev_oot(dev); rng = np.random.default_rng(0)
base = oot.sample(5000, random_state=1)[sel["features"]].reset_index(drop=True)
miss = base.copy()
for c in sel["features"]: miss[c] = miss[c].mask(rng.random(len(miss)) < 0.2)
unk = base.head(500).copy()
for c in S.CAT: unk[c] = unk[c].astype(object); unk.loc[unk.index[:250], c] = "KATEGORI_BARU"          # kategori yang tidak ada di training
test_in = pd.concat([base.assign(kasus="bersih"), miss.assign(kasus="miss20"), unk.assign(kasus="kategori_baru")], ignore_index=True)
for c in S.CAT: test_in[c] = test_in[c].astype(object)
inp = f"{BUNDLE}/_parity_input.parquet"; test_in.to_parquet(inp, index=False)

def reference(df):            # model asli; kategori tak dikenal diubah NaN dengan cara yang sama
    d = df[sel["features"]].copy()
    for c in S.CAT: d[c] = pd.Categorical(d[c].where(d[c].isin(schema["features"][sel["features"].index(c)]["categories"])), categories=dev[c].cat.categories)
    return clf.predict_proba(d)[:, 1]
ref_raw = reference(test_in)
r = subprocess.run([sys.executable, "-I", "deploy/parity_check.py", BUNDLE, inp, "/tmp/_parity_out.npz"], capture_output=True, text=True)
print(r.stdout.strip(), r.stderr.strip()[-500:])
out = np.load("/tmp/_parity_out.npz"); diff = np.abs(out["raw"] - ref_raw)
by_case = pd.DataFrame({"kasus": test_in.kasus, "selisih": diff}).groupby("kasus").selisih.agg(["max", "mean"])
print("selisih skor bundel vs model asli:"); display(by_case)
print("distribusi pita (bundel):", pd.Series(out["band"]).value_counts().to_dict(), "| kualitas rendah:", int((out["quality"] == "low").sum()))
assert diff.max() < 1e-6, "paritas gagal"
lat = json.loads(r.stdout.strip().splitlines()[-1])
json.dump(dict(max_abs_diff=float(diff.max()), cases=by_case.round(9).to_dict(), latency_single_request=lat, n_rows=int(len(test_in))), open(f"{BUNDLE}/parity_report.json", "w"), indent=1)
os.remove(inp); lat

{"n_requests": 200, "p50_ms": 39.4, "p95_ms": 75.9, "max_ms": 100.1} 
selisih skor bundel vs model asli:


,max,mean
kasus,,
bersih,0.0,0.0
kategori_baru,0.0,0.0
miss20,0.0,0.0


distribusi pita (bundel): {'genuine': 10377, 'suspicious': 74, 'high_risk': 49} | kualitas rendah: 73


{'n_requests': 200, 'p50_ms': 39.4, 'p95_ms': 75.9, 'max_ms': 100.1}

In [5]:
# contoh satu request: sebagian fitur tidak ada (None) -> tetap dapat skor, ditandai kualitas rendah, plus reason code
sys.path.insert(0, BUNDLE); from score import FraudScorer
sc = FraudScorer(BUNDLE)
hit = oot[oot[S.LABEL] == 1].iloc[3][sel["features"]]
rec = {k: (None if pd.isna(v) else (v if isinstance(v, str) else float(v))) for k, v in hit.items()}
for k in GROUPS["payee"]: rec[k] = None                    # simulasikan: sumber data payee tidak tersedia
res = sc.score_record(rec); {k: res[k] for k in ("score", "band", "quality", "missing_share", "missing_groups", "reasons")}

{'score': 0.010152456350624561,
 'band': 'suspicious',
 'quality': 'low',
 'missing_share': 0.1618,
 'missing_groups': ['payee'],
 'reasons': ['out_1_vs_avg30=4.455275535583496 (+1.58)',
  'new_device_7=1.0 (+1.07)',
  'night_share_1=0.2845384478569031 (+0.78)']}

## 4. Model card

In [6]:
card = f'''# Model card - fraud risk score v1 (XGB `mask`)
**Status:** kandidat juara, dilatih dan divalidasi pada data **sintetis** (SAS Hackathon 2026). Belum diuji pada data TEST dan belum divalidasi pada data bank.

## Tujuan dan penggunaan
Memberi skor risiko fraud (0-1, terkoreksi prior) untuk satu nasabah-hari dan pita `genuine / suspicious / high_risk`.
Dimaksudkan sebagai **salah satu input** fraud gate di decision flow (proposal §6.3), bukan keputusan tunggal. Skor model saja tidak boleh menurunkan limit nasabah; hanya gate real-time dengan bukti langsung.

## Data dan fitur
* Training: {len(S.cv_pool(dev)):,} nasabah-hari (DEV Okt 2025-Jun 2026, negatif di-down-sample 3%); {len(sel['features'])} fitur (profil, perilaku 1/7/30/90 hari, device, payee, merchant, geo, pass-through).
* Dikeluarkan sengaja: `fraud_alerts_90`, `disputes_90` (bocor sirkular), `near_limit_days_90`, `declined_cnt_90`, `baseline_tier` (bergantung pada limit saat ini).
* Resep pelatihan `mask`: data asli + satu salinan yang 0-40% selnya dikosongkan acak, agar model tahan terhadap fitur tidak lengkap.

## Hasil (PR-AUC berbobot, prevalensi asli ~0,035%)
| Kondisi | CV 5-fold | Uji maju | DEV-OOT Jul-Sep |
|---|---|---|---|
| Bersih | 0,762 | 0,726 | 0,654 (CI95 0,557-0,777) |
| 20% nilai hilang | 0,589 | 0,568 | 0,506 (CI95 0,415-0,602) |
| Tanpa fitur payee | 0,498 | 0,529 | 0,399 |
Stabilitas antar seed (DEV-OOT): bersih 0,669 ± 0,015; 20% hilang 0,532 ± 0,017. Baseline rule gate §6.3: ~0,009. Logistic Regression: 0,209.

## Ambang pita (pada `score`)
suspicious >= {schema['thresholds']['suspicious']:.5f} (presisi >= 5% di DEV-OOT); high_risk >= {schema['thresholds']['high_risk']:.5f} (presisi >= 50%).

## Keterbatasan
* Data sintetis: angka hanya memvalidasi pipeline. Seleksi, ambang, dan klaim performa harus diulang pada data bank (atau sampel anonim) sebelum dipakai.
* Fraud *stealth* (sinyal lemah) tetap lebih sulit; tanpa fitur payee performa turun sekitar 40%.
* Biaya augmentasi: PR-AUC bersih DEV-OOT lebih rendah ~0,04 dibanding model tanpa augmentasi (0,699), sebagai harga ketahanan terhadap data tidak lengkap.
* Model dilatih pada pool tanpa DEV-OOT; belum dilatih ulang pada seluruh data.

## Pemantauan yang disarankan
Tingkat kekosongan fitur per grup, distribusi skor/pita (PSI > 0,25 -> retrain), KS/PR-AUC pada label yang matang, laju alert per kelompok usia dan gender, proporsi `quality = low`.
'''
open(f"{BUNDLE}/MODEL_CARD.md", "w").write(card)
sorted(os.listdir(BUNDLE))

['MODEL_CARD.md',
 '__pycache__',
 'model.json',
 'parity_report.json',
 'requirements.txt',
 'schema.json',
 'score.py']

## Ringkasan
* Juara resmi: **XGB `mask`** (override terdokumentasi); challenger: HGB `mask+grup+noise`.
* Bundel portabel di `bundle/fraud_xgb_mask_v1/` lulus uji paritas (selisih skor di bawah 1e-6, termasuk fitur kosong dan kategori tak dikenal) dan latensi per request tercatat di `parity_report.json`.
* Langkah berikutnya: `03_final_test.ipynb` mengevaluasi **bundel ini** pada data TEST, sekali.